# stardelt quickstart

This notebook is already wired to the platform: it queries Trino **as the person signed in to
JupyterHub** — not as a service account, and with no credential stored in this notebook or in the image.

* **Your identity.** JupyterHub signs you in through Keycloak and hands this server *your* access token
  when it starts (`TRINO_TOKEN`). Trino checks that token against the realm's keys and runs every
  statement as the identity inside it.
* **Your catalog.** The next cells pick the catalog this account may write to — `shared` for a member,
  `playground` in the public demo — so the notebook runs for whoever gets it. The table it creates stays;
  nothing here drops a table, because drop and rename are platform-administrator actions in this release.

In [ ]:
import base64
import json
import os
from urllib.parse import urlsplit

import pandas as pd
from trino.auth import JWTAuthentication
from trino.dbapi import connect

TRINO_URL = os.environ.get("TRINO_URL", "https://trino.alpha.dev.stardelt.io:443")
TOKEN = os.environ.get("TRINO_TOKEN")
HUB_USER = os.environ.get("JUPYTERHUB_USER")


def token_claims(token):
    """The claims inside the JWT, without verifying it (Trino verifies)."""
    if not token or token.count(".") != 2:
        return {}
    payload = token.split(".")[1]
    payload += "=" * (-len(payload) % 4)
    return json.loads(base64.urlsafe_b64decode(payload))


CLAIMS = token_claims(TOKEN)
# The engine user Trino derives from the token: the username
# (`preferred_username`) before stardelt-platform#311, the Keycloak user id
# (`sub`) after it. This notebook never declares one — a declared name that
# differs from the token's is refused ("user X cannot impersonate user Y") — so
# it runs with either setting.
IDENTITIES = {value for value in (CLAIMS.get("sub"), CLAIMS.get("preferred_username")) if value}

# Over HTTPS Trino accepts the bearer token as-is, so the notebook needs no
# password and no secret: the hub copies the signed-in user's token in at spawn.
_url = urlsplit(TRINO_URL)
CONNECT = dict(
    host=_url.hostname,
    port=_url.port or (443 if _url.scheme == "https" else 8080),
    http_scheme=_url.scheme,
    source="stardelt-warehouse-quickstart",
)
if TOKEN:
    CONNECT["auth"] = JWTAuthentication(TOKEN)
else:
    # Nothing to authenticate with. Name a user anyway so the statement reaches
    # Trino and comes back 401 — the identity cell below proves that too.
    CONNECT["user"] = HUB_USER or "notebook"
conn = connect(**CONNECT)


def sql(query):
    """Run `query` and return its rows as a pandas DataFrame."""
    cur = conn.cursor()
    cur.execute(query)
    columns = [d[0] for d in (cur.description or [])]
    return pd.DataFrame(cur.fetchall(), columns=columns)


print("Trino:    ", TRINO_URL)
print("Hub user: ", HUB_USER or "(not running under JupyterHub)")
print("Engine user from the token:", ", ".join(sorted(IDENTITIES)) or "(none)")
if not TOKEN:
    print("TRINO_TOKEN is not set — the queries below will be refused with 401.")

In [ ]:
me = sql("SELECT current_user AS signed_in_as")
current_user = me.loc[0, "signed_in_as"]
print(me.to_string(index=False))

if HUB_USER:
    assert CLAIMS.get("preferred_username") == HUB_USER, (
        f"The token in TRINO_TOKEN belongs to {CLAIMS.get('preferred_username')!r}, "
        f"but the hub user is {HUB_USER!r}"
    )
    assert current_user in IDENTITIES, (
        f"Trino authenticated this session as {current_user!r}, which is neither the "
        f"`sub` nor the `preferred_username` of your token"
    )
    print(f"\nOK: Trino runs as {current_user!r} — the token in TRINO_TOKEN is yours.")

In [ ]:
catalogs = sorted(sql("SHOW CATALOGS")["Catalog"])
candidates = [name for name in ("shared", "playground") if name in catalogs]
assert candidates, (
    "this account may use neither `shared` nor `playground` — ask an administrator for a grant "
    f"(catalogs this account can see: {', '.join(catalogs)})"
)
CATALOG = candidates[0]
print(f"Catalog: {CATALOG}")
print("Visible catalogs:", ", ".join(catalogs))

In [ ]:
import re

print(sql(f"SHOW SCHEMAS FROM {CATALOG}").to_string(index=False))

# A schema named after you, inside a catalog your group may write. Two people in the same group
# share the schema; in the public demo everyone is `demo`, so a visitor sees another visitor's table.
SCHEMA = re.sub(r"[^a-z0-9_]", "_", (HUB_USER or current_user or "you").lower())
sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
print(f"\n{CATALOG}.{SCHEMA} is ready.")

In [ ]:
# The helper package this image ships (the stardelt library, stardelt/stardelt-python, pinned to
# one release): the same catalog through two calls, without the connection setup above.
import stardelt

me = stardelt.sql("SELECT current_user AS signed_in_as")
assert me.loc[0, "signed_in_as"] == current_user, me
print(f"stardelt.sql runs as {current_user!r} — the same identity as above.")

# Governed write: a 3-row frame round-trips through Trino. Re-running this cell is safe — the table
# is created if it is missing and emptied first. Nothing here drops a table: since platform#1130 a
# `write` grant is Lakekeeper `write`, and drop/rename are platform-administrator actions.
TABLE = f"{CATALOG}.{SCHEMA}.quickstart_smoke"
sql(f"CREATE TABLE IF NOT EXISTS {TABLE} (id INTEGER, note VARCHAR)")
sql(f"DELETE FROM {TABLE}")
smoke = pd.DataFrame({"id": [1, 2, 3], "note": ["a", "b", "c"]})
stardelt.write_df(smoke, TABLE, mode="append")
back = stardelt.sql(f"SELECT id, note FROM {TABLE} ORDER BY id")
pd.testing.assert_frame_equal(back, smoke)
print(f"{TABLE}: {len(back)} rows written and read back — re-run this cell any time.")

## What just happened

`SELECT current_user` is your identity from your own token — never a shared account. That is the
platform's "one identity" rule in practice: the same token that opened this notebook is the token Trino
authorises, so row and column policy (when it is configured) applies to your queries, and the audit trail
names you.

The schema you created is Apache Iceberg, served by Lakekeeper, with Parquet on the platform's S3. Every
engine that speaks this catalog sees the same tables — and the same permissions, because the policy
attaches to the token, not to the tool.

The table stays. To rebuild it, `CREATE OR REPLACE TABLE` is the operation a `write` grant covers;
dropping is a platform-administrator action in this release.

In [ ]:
# Spark Connect — the catalogs you may see, with your token. Spark is heavy compute and optional in an
# install (`make install-spark`); when it is missing this cell says what it found instead of failing the
# notebook. Point SPARK_REMOTE elsewhere through the hub's singleuser.extraEnv if you moved it.
import socket
from urllib.parse import urlsplit

import stardelt

SPARK_REMOTE = os.environ.get(
    "SPARK_REMOTE", "sc://spark-connect.stardelt.svc.cluster.local:15002"
)
_spark = urlsplit(SPARK_REMOTE.replace("sc://", "http://"))
try:
    # A short TCP probe first: a client pointed at a blackholed endpoint retries for a long time,
    # and a quickstart should not hang.
    socket.create_connection((_spark.hostname, _spark.port or 15002), timeout=2).close()

    # The helper asks Trino which catalogs you may see and configures them with your token;
    # Lakekeeper decides what the session may read.
    spark = stardelt.spark()
    print("connected to Spark Connect at", SPARK_REMOTE)
    rows = spark.sql(f"SELECT count(*) AS rows FROM {TABLE}")
    rows.show(truncate=False)
    print(f"Spark read {TABLE} through a second engine: {rows.first()['rows']} rows")
except Exception as exc:  # noqa: BLE001 - Spark is an optional component
    print(f"Spark Connect is not reachable at {SPARK_REMOTE}: {type(exc).__name__}: {exc}")
    print("Known causes: Spark is not installed (`make install-spark`), the singleuser")
    print("NetworkPolicy lacks the egress rule (helm-values/jupyterhub.yaml), or the")
    print("token predates `make sync-catalog-audience` — restart the server and sign in again.")

### Large results and the shared driver

Every notebook shares one Spark driver (1 GiB heap, 3 GiB pod), so keep results out of it:

| Instead of | Do |
|---|---|
| `df.toPandas()` / `df.collect()` on a big table | `df.limit(10_000).toPandas()`, aggregate first, or `df.writeTo(...)` a table and read it with SQL |
| `spark.createDataFrame(big_pandas_or_list)` | write it to a table once (`writeTo`), then `spark.table(...)` |
| a Python loop that runs one job per item | one query with a join or `groupBy`, or `repartition(n)` + `mapInPandas` |
| `df.join(small_df, ...)` repeated in a loop | `broadcast(small_df)` once, outside the loop |
| `df.cache()` never released | `df.unpersist()` when done; cache only what several actions reuse |
| 250,000-row chunks into `createDataFrame` + `saveAsTable` per chunk | chunks of at most 20,000 rows, finished chunk numbers in a small table (platform#378) |
| repeated `df.count()` | run it once, keep the number |

A result above `spark.driver.maxResultSize` (512 MiB) fails the call: write the result to a table instead of raising the limit. Executors are at most 2 x 1 core x 1 GiB today. `stardelt.spark_tips()` prints the same rows.


### Spark Connect: RDDs and `sparkContext`

There is no `sparkContext` and no RDD API here: both live in the JVM on the other side of
the connection, and Spark Connect does not expose them. `df.rdd` raises `[NOT_IMPLEMENTED]`
with a sentence that names the replacement; the rest is this table.

| RDD API | On Spark Connect |
|---|---|
| `df.rdd`, `df.rdd.getNumPartitions()` | `df.select(spark_partition_id()).distinct().count()` |
| `rdd.map(...)` | `df.select(...)` / `df.withColumn(...)`, or `df.mapInPandas(...)` for per-partition Python |
| `rdd.foreach(...)` | `df.foreach(...)` (runs your Python on a worker), or write the result to a table |
| `sparkContext.parallelize(...)` | `spark.createDataFrame(...)`, `spark.range(...)` |
| `sparkContext.broadcast(x)` | a join with `F.broadcast(small_df)` |
| accumulators | `groupBy(...).agg(...)` |
| `sparkContext.setLogLevel(...)` | not available — the driver's level is shared by every notebook |

Extra libraries are not on the executors either: only this image's pandas, pyarrow and numpy
are there (the Python on executors section below names the versions).


### Python on executors

The Spark cell above is a DataFrame read, and that never runs your Python on a worker. UDFs
do: `udf`, `pandas_udf`, `mapInPandas`, `applyInPandas`, `foreach` and `foreachPartition`
send your function to the executors, where CPython **3.11** — the same minor version as
this notebook — runs it with exactly pandas 2.2.3, pyarrow 18.1.0 and numpy 2.0.2. Any
other library you install in this notebook is *not* on the executor: an `import` of it
inside a UDF fails there, so keep that work in the notebook, where the environment is
yours.


## What this notebook can and cannot do today

* **Can:** `SHOW` and `SELECT` against every catalog Trino exposes to you, with your own identity — the
  engine's own `system`, `tpch` and `tpcds` catalogs are for administrators — write inside the catalog
  your group was granted (`shared` for a member, `playground` in the public demo) and read the same
  Iceberg tables through Spark Connect, a second engine.
* **Cannot:** drop or rename a table. Since platform#1130 that needs Lakekeeper's `modify`, which only
  platform administrators hold; a `write` grant covers `CREATE OR REPLACE`, `INSERT`, `UPDATE`, `DELETE`,
  `MERGE`, `ALTER` and `TRUNCATE`.
* **TLS, in-cluster.** With the platform IdP installed, `TRINO_URL` is the coordinator's in-cluster HTTPS
  endpoint (`https://trino.stardelt.svc.cluster.local:8443`) and `REQUESTS_CA_BUNDLE` is the internal CA
  that signs it — a real anchor, not a disabled certificate check. Nothing goes through the public
  ingress, and a plain-HTTP statement is refused, so the token is the only way in.
* **Token lifetime:** the hub's `jupyterhub` client issues an 8-hour token, placed in this server's
  environment at spawn. When it expires, restart the server from the hub — the hub refreshes your token
  before every spawn. After the Keycloak SSO session ends (10 hours), sign in again.
* **Spark:** the cell above opens a session against the ClusterIP-only Spark Connect endpoint and reads
  the table this notebook wrote, through the catalogs Trino lists for you, with your token. Spark is
  optional: without `make install-spark` the cell says what it found.
* **The helper:** `import stardelt` (shipped in the image, from the pinned release of
  `stardelt/stardelt-python`) gives `sql` and `write_df` — the queries above without the setup — plus
  `spark()` for bulk work. Catalog access is your identity; the S3 key stays the platform's, and row and
  column masks are Trino-only (design `multitenant-catalogs.md` §5.8).